# 03 Embeddings and Language Model Inputs

## Problem

token id 为什么不能直接送进神经网络？embedding table 到底是什么，它和查表、线性代数、语义空间之间到底是什么关系？

## Dependencies

建议先完成 `02_tokenization_and_bpe.ipynb`，理解 token 和 token id 是怎么来的。


## Goals

- 理解离散 token id 与连续向量表示的根本差别
- 理解 embedding lookup 的本质就是“按索引取一行参数”
- 说清楚 `(batch, seq_len)` 与 `(batch, seq_len, hidden_dim)` 这两类 shape 的意义
- 能把“文本 -> token ids -> embeddings”这条路径真正讲明白

## Scope and Approach

这一节重点不是展示一张 embedding table，而是解决一个非常常见的误区：很多人知道 embedding 是“向量表示”，但不知道为什么 token id 本身不能直接训练，也不知道 embedding lookup 和普通参数矩阵之间到底是什么关系。


## 为什么 token id 不够

token id 只是一个索引，不是带语义的数值。

比如：

- `deep -> 1`
- `seek -> 2`
- `math -> 4`

这里的 `1`、`2`、`4` 没有任何自然的大小关系。`4` 并不比 `2` “更数学”，`2` 也不比 `1` “更接近 deep”。它们只是词表里的编号。

如果把 token id 直接送进网络，模型就会被迫把这种毫无语义的编号顺序也一起学进去，这是错误的输入表示。

embedding 的作用，就是把这个离散编号空间，映射到一个连续向量空间里。


In [ ]:
import numpy as np

np.random.seed(42)
np.set_printoptions(precision=3, suppress=True)

token_to_id = {
    '<bos>': 0,
    'deep': 1,
    'seek': 2,
    'likes': 3,
    'math': 4,
}

vocab_size = len(token_to_id)
hidden_dim = 6

# embedding_table 可以直接理解成一张参数表：
#   行数 = 词表大小 vocab_size
#   列数 = 每个 token 的表示维度 hidden_dim
embedding_table = np.random.randn(vocab_size, hidden_dim) * 0.1

# 这里 input_ids 的 shape = (batch, seq_len)
# 2 表示 batch size = 2
# 4 表示每条样本有 4 个 token 位置
input_ids = np.array([
    [0, 1, 2, 4],
    [0, 2, 3, 4],
])

print('input_ids.shape =', input_ids.shape)
print('embedding_table.shape =', embedding_table.shape)


## Embedding table 到底是什么

embedding table 可以先非常直接地理解成一张参数表：

- 每个 token 对应一行
- 每一行是一个长度为 `hidden_dim` 的向量
- 训练时这些向量会被不断更新

所以 embedding lookup 并不神秘，它只是：

- 给我 token id `3`
- 我去参数矩阵的第 3 行把那一行取出来

这一步之后，输入就从整数索引变成了连续向量。


In [ ]:
# 这一步就是最核心的 embedding lookup：
# 用 input_ids 作为索引，从 embedding_table 中取出对应的行。
embedded = embedding_table[input_ids]

# 结果 shape 会变成 (batch, seq_len, hidden_dim)
# 也就是：
#   每条样本的每个位置
#   都不再是一个整数 id
#   而是一个 hidden_dim 维向量
print('embedded.shape =', embedded.shape)
print('第一条样本的 embedding 序列 =')
print(embedded[0])

print('token id 1 对应向量 =', embedding_table[1])
print('token id 2 对应向量 =', embedding_table[2])


## 随机初始化的向量怎么会有语义

很多人会问：embedding 一开始不是随机的吗？那它为什么后来会带语义？

答案是：

- 初始化时它当然没有语义
- 但在训练中，如果两个 token 常常出现在相似上下文里，它们的向量就会被推向更接近的位置
- 如果某个 token 总在完全不同的上下文里出现，它的向量就会被推向不同区域

所以 embedding 的语义不是人工写进去的，而是通过训练目标慢慢形成的。

它本质上是在回答一个问题：

**什么样的连续表示，最有利于模型在后面把下一个 token 预测对？**


In [ ]:
# 用最简单的点积观察 token 之间的关系。
# 这里只是演示形式，不代表随机初始化时真的已经有稳定语义。
vec_deep = embedding_table[token_to_id['deep']]
vec_seek = embedding_table[token_to_id['seek']]
vec_math = embedding_table[token_to_id['math']]

print('deep · seek =', np.dot(vec_deep, vec_seek))
print('deep · math =', np.dot(vec_deep, vec_math))

# 这里再加一个非常朴素的位置表，提前说明后面位置编码在做什么：
# token embedding 回答“这是什么 token”
# position embedding / position information 回答“它在第几个位置”
position_table = np.arange(input_ids.shape[1] * hidden_dim).reshape(input_ids.shape[1], hidden_dim) * 0.01
model_input = embedded + position_table[None, :, :]

print('position_table.shape =', position_table.shape)
print('model_input.shape =', model_input.shape)


## 模型真正吃进去的是什么

到这里，模型真正吃进去的已经不是文本，也不是 token id，而是：

- token embedding
- 加上位置相关信息后的输入向量

也就是说，从这一刻开始，LLM 看到的是一串向量，而不是一串词。

后面的 attention、FFN、norm，并不是直接在处理文字，而是在处理这串连续向量。

## Common mistakes

- 把 token id 当成天然有大小顺序的数值。事实上它只是一个字典索引。
- 以为 embedding 只是为了把维度升高。真正关键的是它把离散符号变成可学习的连续表示。
- 把 embedding 和位置编码混为一谈。前者回答“是什么 token”，后者回答“它在第几个位置”。
- 以为 embedding 一开始就有语义。实际上语义是在训练中逐步形成的。

## Checkpoints

- 用自己的话解释为什么 token id 不能直接送进网络。
- 尝试把 `hidden_dim` 改大到 16，重新观察 embedding table 的形状。
- 思考为什么两个语义接近的 token，训练后常常会拥有更接近的向量。
- 说明 `(batch, seq_len)` 和 `(batch, seq_len, hidden_dim)` 分别表示什么。

## Summary

embedding 是语言模型真正意义上的输入层。离散 token id 经过查表，第一次变成了模型可以计算、可以优化、可以逐层变换的连续向量。后面所有 attention、FFN、norm，本质上都是在处理这些向量。

## Next Module

下一节开始进入 self-attention。也就是 Transformer 最核心、也最值得真正拆透的一块。
